# CSV - Value Errors

Introduce declared-range violations, incorrect numeric scaling, or both, across scattered and grouped observations.

**Instructions:** Use `prepare_data_csv_update.ipynb` to create a clean `data/weather.csv` first. This notebook overwrites it; restore clean data before rerunning. Define a meaningful valid range and unit convention for the chosen column. Unusual values alone are not proof of an error. Restore clean input for every run; plausible scaled values cannot always be recognized as previously corrupted.

In [ ]:
from pathlib import Path
from random import Random
import pandas as pd

# Configuration
DATA_PATH = Path("../../data/weather.csv")
TIME_COLUMN = "time"
TIMEZONE = "America/Edmonton"
START_DATE = "2025-06-01"
END_DATE = "2025-09-01"  # Exclusive; local calendar dates.
SEED = 814205

# Value Errors
ERROR_TARGET_COLUMN = "relative_humidity_2m"
ERROR_KIND = "mixed"  # "out_of_bounds", "scaled", "mixed", "random".
ERROR_PLACEMENT = "mixed"  # "scattered", "grouped", "mixed", "random".
ERROR_RATE = (0.01, 0.03)  # Share of current eligible nonmissing/valid values changed.
ERROR_GROUPS = 3
ERROR_GROUPED_SHARE = 0.70
ERROR_BOUNDS_SHARE = 0.50  # In mixed kind, share forced outside the valid range.
VALID_MIN = 0
VALID_MAX = 100
OUTSIDE_MARGIN = 10.0  # Values forced at least this far outside the range.
SCALE_FACTOR = 1.5  # A wrong measurement or unit-conversion factor.


In [ ]:
# Shared placement helper: copy once into a combined dirty_data.py.
def select_positions(n, rng, *, method, rate, n_groups, grouped_share):
    """Select a fraction of the current eligible records in ordered positions.

    Grouped blocks have at least two consecutive eligible records; mixed
    placement reserves at least one independently scattered selection.
    """
    if n < 1:
        raise ValueError("No eligible records.")
    if method not in ("scattered", "grouped", "mixed", "random"):
        raise ValueError("Unknown placement method.")
    if not 0 < rate[0] <= rate[1] < 1:
        raise ValueError("Rate must be between 0 and 1.")
    if n_groups < 1 or not 0 < grouped_share < 1:
        raise ValueError("Invalid group count or grouped share.")

    chosen = rng.choice(("scattered", "grouped", "mixed")) if method == "random" else method
    count = max(1, round(n * rng.uniform(*rate)))
    if chosen == "mixed" and count < 3:
        raise ValueError("Mixed placement needs two grouped and one scattered record.")
    grouped_count = (count if chosen == "grouped" else
                     max(2, min(count - 1, round(count * grouped_share))) if chosen == "mixed" else 0)
    selected = set()
    if grouped_count:
        if grouped_count < 2:
            raise ValueError("Grouped placement requires at least two affected records.")
        groups = min(n_groups, grouped_count // 2)
        if n - grouped_count < groups - 1:
            raise ValueError("Insufficient eligible records for separate groups.")
        length, extra = divmod(grouped_count, groups)
        lengths = [length + (i < extra) for i in range(groups)]
        rng.shuffle(lengths)
        slack = n - grouped_count - (groups - 1)
        dividers = sorted(rng.sample(range(slack + groups), groups))
        gaps = ([dividers[0]] +
                [dividers[i] - dividers[i - 1] - 1 for i in range(1, groups)] +
                [slack + groups - 1 - dividers[-1]])
        cursor = gaps[0]
        for i, block_length in enumerate(lengths):
            selected.update(range(cursor, cursor + block_length))
            if i < groups - 1:
                cursor += block_length + 1 + gaps[i + 1]

    scattered_count = count - grouped_count
    if scattered_count:
        remaining = [i for i in range(n) if i not in selected]
        selected.update(rng.sample(remaining, k=scattered_count))
    if len(selected) != count:
        raise AssertionError("Selected count differs from the intended count.")
    return sorted(selected), chosen

In [ ]:
def add_value_errors(
    df, rng, *, time_column, timezone, start_date, end_date,
    target_column, kind, placement, error_rate, n_groups, grouped_share,
    bounds_share, valid_min, valid_max, outside_margin, scale_factor,
):
    if time_column not in df or target_column not in df:
        raise ValueError("Time or target column not found.")
    if not pd.api.types.is_numeric_dtype(df[target_column]):
        raise ValueError("This numeric error function requires a numeric column.")
    if valid_min >= valid_max or outside_margin <= 0:
        raise ValueError("Provide valid lower/upper bounds and a positive outside margin.")
    if scale_factor == 1 or scale_factor == 0:
        raise ValueError("SCALE_FACTOR must differ from 0 and 1.")

    dirty = df.reset_index(drop=True).copy()
    time = pd.to_datetime(dirty[time_column], utc=True)
    if time.isna().any():
        raise ValueError("Timestamps cannot be missing.")
    start = pd.Timestamp(start_date, tz=timezone).tz_convert("UTC")
    end = pd.Timestamp(end_date, tz=timezone).tz_convert("UTC")
    if start >= end:
        raise ValueError("Start date must precede end date.")
    if kind not in ("out_of_bounds", "scaled", "mixed", "random"):
        raise ValueError("Unrecognized error kind.")
    if placement not in ("scattered", "grouped", "mixed", "random"):
        raise ValueError("Unrecognized error placement.")
    if not 0 < error_rate[0] <= error_rate[1] < 1:
        raise ValueError("Error rate must lie between 0 and 1.")
    if not 0 < grouped_share < 1 or not 0 < bounds_share < 1:
        raise ValueError("Shares must lie between 0 and 1.")

    selected_kind = rng.choice(("out_of_bounds", "scaled", "mixed")) if kind == "random" else kind
    selected_placement = rng.choice(("scattered", "grouped", "mixed")) if placement == "random" else placement

    # Existing nulls are not altered. Scaling excludes zeros because 0 × factor stays 0.
    scope = time.ge(start) & time.lt(end)
    valid = dirty[target_column].between(valid_min, valid_max)
    if (scope & dirty[target_column].notna() & ~valid).any():
        raise ValueError("Existing out-of-range values detected. Restore clean data first.")
    eligible_mask = scope & dirty[target_column].notna() & valid
    if selected_kind in ("scaled", "mixed"):
        eligible_mask &= dirty[target_column].ne(0)
    eligible = dirty.loc[eligible_mask].sort_values(time_column, kind="stable")
    if eligible.empty:
        raise ValueError("No eligible valid values; check the column, bounds, and dates.")

    selected, _ = select_positions(
        len(eligible), rng, method=placement, rate=error_rate,
        n_groups=n_groups, grouped_share=grouped_share,
    )
    count = len(selected)
    if selected_kind == "mixed" and count < 2:
        raise ValueError("Mixed error kinds require at least two values.")

    rows = eligible.index[selected]
    bound_count = (count if selected_kind == "out_of_bounds" else 0 if selected_kind == "scaled" else
                   max(1, min(count - 1, round(count * bounds_share))))
    bound_rows = rng.sample(list(rows), k=bound_count)
    bound_set = set(bound_rows)
    scaled_rows = [i for i in rows if i not in bound_set]
    previous_values = dirty.loc[rows, target_column].copy()
    if pd.api.types.is_integer_dtype(dirty[target_column]):
        dirty[target_column] = dirty[target_column].astype(float)

    if bound_rows:
        # Every assigned value violates a declared numeric limit.
        for idx in bound_rows:
            dirty.at[idx, target_column] = (valid_max + outside_margin + rng.random() * outside_margin
                                             if rng.random() < 0.5 else
                                             valid_min - outside_margin - rng.random() * outside_margin)
    if scaled_rows:
        dirty.loc[scaled_rows, target_column] = dirty.loc[scaled_rows, target_column] * scale_factor

    if not dirty.loc[rows, target_column].ne(previous_values).all():
        raise AssertionError("An intended value error failed to change its target.")
    return dirty


In [ ]:
# Small business-like trial: a discount percentage must stay between 0 and 100.
trial = pd.DataFrame({
    "time": pd.date_range("2025-07-01 06:00", periods=120, freq="2h", tz="UTC"),
    "discount_pct": [float(5 + i % 70) for i in range(120)],
    "transaction_id": range(120),
})
trial_dirty = add_value_errors(
    trial, Random(SEED), time_column="time", timezone=TIMEZONE,
    start_date="2025-07-01", end_date="2025-08-01",
    target_column="discount_pct", kind="mixed", placement="mixed",
    error_rate=(0.10, 0.10), n_groups=2, grouped_share=0.70,
    bounds_share=0.50, valid_min=0, valid_max=100,
    outside_margin=10, scale_factor=1.5,
)
assert len(trial_dirty) == len(trial)
assert trial_dirty["transaction_id"].equals(trial["transaction_id"])
assert trial_dirty["discount_pct"].ne(trial["discount_pct"]).sum() == 12
trial_dirty.loc[trial_dirty["discount_pct"].ne(trial["discount_pct"])].head()


In [ ]:
# Apply to a freshly prepared CSV; overwrite weather.csv.
clean = pd.read_csv(DATA_PATH)
dirty = add_value_errors(
    clean, Random(SEED), time_column=TIME_COLUMN, timezone=TIMEZONE,
    start_date=START_DATE, end_date=END_DATE,
    target_column=ERROR_TARGET_COLUMN, kind=ERROR_KIND, placement=ERROR_PLACEMENT,
    error_rate=ERROR_RATE, n_groups=ERROR_GROUPS, grouped_share=ERROR_GROUPED_SHARE,
    bounds_share=ERROR_BOUNDS_SHARE, valid_min=VALID_MIN, valid_max=VALID_MAX,
    outside_margin=OUTSIDE_MARGIN, scale_factor=SCALE_FACTOR,
)
dirty.to_csv(DATA_PATH, index=False)
saved = pd.read_csv(DATA_PATH)


In [ ]:
# Inspect: missingness checks alone won't reveal incorrect values.
saved[ERROR_TARGET_COLUMN].describe()


In [ ]:
saved.loc[~saved[ERROR_TARGET_COLUMN].between(VALID_MIN, VALID_MAX), [TIME_COLUMN, ERROR_TARGET_COLUMN]].head(20)


In [ ]:
# Verify changed targets and unchanged surrounding data after CSV round-trip.
changed = saved[ERROR_TARGET_COLUMN].ne(clean[ERROR_TARGET_COLUMN])
assert changed.any()
assert len(saved) == len(clean)
pd.testing.assert_frame_equal(
    saved.drop(columns=ERROR_TARGET_COLUMN), clean.drop(columns=ERROR_TARGET_COLUMN),
    check_exact=False, rtol=1e-12,
)
pd.testing.assert_series_equal(
    saved.loc[~changed, ERROR_TARGET_COLUMN], clean.loc[~changed, ERROR_TARGET_COLUMN],
    check_dtype=False, check_exact=False, rtol=1e-12,
)
print("Values changed:", int(changed.sum()))
print("Outside declared bounds:", int((~saved[ERROR_TARGET_COLUMN].between(VALID_MIN, VALID_MAX)).sum()))


In [ ]:
import matplotlib.pyplot as plt

# Developer-side comparison: only changed values are overlaid on the clean data.
local_time = pd.to_datetime(clean[TIME_COLUMN], utc=True).dt.tz_convert(TIMEZONE)
start = pd.Timestamp(START_DATE, tz=TIMEZONE)
end = pd.Timestamp(END_DATE, tz=TIMEZONE)
scope = local_time.ge(start) & local_time.lt(end)
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.scatter(local_time.loc[scope], clean.loc[scope, ERROR_TARGET_COLUMN], s=5, alpha=0.4, label="Clean")
ax.scatter(local_time.loc[changed], saved.loc[changed, ERROR_TARGET_COLUMN], s=12, label="Altered")
ax.axhline(VALID_MIN, linestyle="--", alpha=0.6)
ax.axhline(VALID_MAX, linestyle="--", alpha=0.6)
ax.set_title(f"Introduced value errors: {ERROR_TARGET_COLUMN}")
ax.set_xlabel("Local date")
ax.set_ylabel(ERROR_TARGET_COLUMN)
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()
